# 2024—2025 历史披露篮子分钟源

固定全期、实际历史成员、原证券身份。仅采集与原生读回；没有训练、收益筛选或超额结论。

In [1]:
from pathlib import Path
import os,json,hashlib,threading
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
import exchange_calendars as xc
from src.data.minute_intake import collect_futu_bundle,atomic_json
from src.data.minute_price_basis import normalize_none_history
from src.data.semis_context_refresh import audit_context
from src.data.soxx_pit_holdings import build_pit_soxx_weight_panel
from src.data.qlib_native import build_qlib_store_from_frame,load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/constituent_intraday_history_20261014';plan=json.loads((card/'plan.json').read_text());output=root/plan['raw_directory'];output.mkdir(parents=True,exist_ok=True)
lock=threading.Lock()
def progress(value):
 with lock:atomic_json(value,Path(plan['progress_path']))
hp=root/plan['holdings_path'];assert hashlib.sha256(hp.read_bytes()).hexdigest()==plan['holdings_sha256'];holdings=pd.read_csv(hp)
schedule=xc.get_calendar('XNYS',start=plan['start'],end=plan['as_of']).schedule
weights=build_pit_soxx_weight_panel(holdings,calendar=schedule.index,availability_lag_days=plan['availability_lag_days']);assert weights.groupby('datetime').size().eq(30).all();assert pd.to_datetime(weights.filing_date).lt(weights.datetime).all()
source_bundles={};entries=[];raw_sources={};rehab={};parts=[];coverages=[]
for group in plan['groups']:
 directory=output/group['directory'];source=collect_futu_bundle(requests={s:'K_5M' for s in group['symbols']},start=group['start'],end=group['end'],output_dir=directory,checkpoint_dir=Path(plan['checkpoint_dir'])/group['directory'],workers=plan['workers'],request_pause_seconds=plan['request_pause_seconds'],progress=progress);assert source['complete'] and source['adjustment']=='NONE' and source['session']=='RTH';source_bundles[group['directory']]=source
 for entry in source['entries']:
  path=directory/entry['path'];assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256'];data=pd.read_parquet(path);raw_sources[entry['symbol']]=data;normalized=normalize_none_history(data,symbol=entry['symbol'],interval='5m',fetched_at=entry['fetched_at']);parts.append(normalized);coverages.append(audit_context(normalized,schedule.loc[group['start']:group['end']],symbols=[entry['symbol']]));entries.append(dict(entry,source_directory=str(directory.relative_to(root)),reused=False))
 for entry in source['rehab']:rehab[entry['symbol']]=pd.read_parquet(directory/entry['path'])
mp=root/plan['reuse_manifest'];assert hashlib.sha256(mp.read_bytes()).hexdigest()==plan['reuse_manifest_sha256'];reuse=json.loads(mp.read_text());assert reuse['complete'] and reuse['adjustment']=='NONE' and reuse['session']=='RTH';source_bundles['reused']=reuse
for entry in reuse['entries']:
 if entry['symbol'] not in plan['reuse_symbols']:continue
 assert entry['ktype']=='K_5M';path=mp.parent/entry['path'];assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256'];data=pd.read_parquet(path);raw_sources[entry['symbol']]=data
 normalized=normalize_none_history(data,symbol=entry['symbol'],interval='5m',fetched_at=entry['fetched_at']);normalized=normalized.loc[normalized.date.between(plan['start'],plan['as_of'])];parts.append(normalized);coverages.append(audit_context(normalized,schedule,symbols=[entry['symbol']]));entries.append(dict(entry,source_directory=str(mp.parent.relative_to(root)),reused=True))
raw=pd.concat(parts,ignore_index=True);coverage=pd.concat(coverages,ignore_index=True);assert raw.is_final.all();assert not raw.duplicated(['symbol','timestamp']).any();assert raw.loc[raw.symbol.eq('WOLF'),'date'].max()<='2024-11-26'
coverage['complete_valid']=coverage.missing_bars.eq(0)&coverage.unexpected_bars.eq(0)&coverage.invalid_rows.eq(0)
active=weights.assign(date=weights.datetime.dt.strftime('%Y-%m-%d')).rename(columns={'instrument':'symbol'}).merge(coverage,on=['symbol','date'],how='left',validate='one_to_one');assert active.complete_valid.notna().all()
basket=active.groupby('date').agg(full_basket=('complete_valid','all'),members=('symbol','size'),missing_bars=('missing_bars','sum'),invalid_rows=('invalid_rows','sum'))
by_symbol=active.groupby('symbol').agg(active_sessions=('date','size'),complete_valid_active_sessions=('complete_valid','sum'),missing_bars=('missing_bars','sum'),invalid_rows=('invalid_rows','sum'))
provider=root/plan['provider'];fields=['open','high','low','close','volume','turnover','price_valid'];manifest=build_qlib_store_from_frame(raw.rename(columns={'symbol':'instrument','timestamp':'datetime'}),provider_uri=provider,freq='5min',as_of=plan['as_of'],field_map={x:x for x in fields},source='Futu NONE RTH actual published historical basket union; daily membership separately gated; no imputation')
qlib.init(provider_uri=str(provider),region='us');native=load_qlib_feature_frame(QlibProviderConfig(provider,freq='5min'),instruments=plan['symbols'],fields=['$open','$close','$volume','$price_valid'],start=plan['start'],end=plan['as_of']+' 16:00:00');assert native.attrs.get('qlib_data_api')==['qlib.data.D.features']
expected=raw.copy();expected['datetime']=expected.timestamp.dt.tz_convert('America/New_York').dt.tz_localize(None);expected=expected.rename(columns={'symbol':'instrument'}).set_index(['instrument','datetime'])
for field in ['open','close','volume','price_valid']:np.testing.assert_allclose(native['$'+field].reindex(expected.index),expected[field],rtol=2e-7,atol=1e-4)
source_manifest={'schema_version':'historical_basket_minute_source_v1','complete':True,'start':plan['start'],'as_of':plan['as_of'],'adjustment':'NONE','session':'RTH','entries':entries,'holdings_sha256':plan['holdings_sha256'],'membership_filter_required':True};atomic_json(source_manifest,output/'source_manifest.json')
summary={'experiment':card.name,'decision':'historical_basket_source_only_no_alpha_claim','new_fits':0,'new_backtests':0,'future_outcome_tests':0,'objective_achieved':False,'symbols':len(plan['symbols']),'raw_bars':len(raw),'sessions':len(basket),'complete_valid_basket_sessions':int(basket.full_basket.sum()),'missing_active_bars':int(active.missing_bars.sum()),'invalid_active_rows':int(active.invalid_rows.sum()),'historical_first_receipt_certified':False,'reused_history_not_fresh_oos':True,'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/coverage','analysis/active_coverage','analysis/native']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='historical_basket_source_intake',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,workers=plan['workers'],request_pause_seconds=plan['request_pause_seconds'])
 rec.save_objects(artifact_path='inputs',plan=plan,holdings=holdings,weights=weights,source_manifest=source_manifest,source_bundles=source_bundles,raw_sources=raw_sources,rehab=rehab,provider_manifest=manifest,quota_before=json.loads((card/'quota_before.json').read_text()))
 rec.save_objects(artifact_path='analysis',coverage=coverage,active_coverage=active,native=native,summary=summary)
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),observed_bars=len(raw))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');by_symbol.to_csv(card/'symbol_coverage.csv');basket.reset_index().to_csv(card/'basket_coverage.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];print({k:summary[k] for k in ['raw_bars','complete_valid_basket_sessions','missing_active_bars','invalid_active_rows']})


In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.data.closed_intraday_window import closed_window_return
from src.data.qlib_native import load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/constituent_intraday_history_20261014';plan=json.loads((card/'plan.json').read_text());extra=json.loads((card/'observation_plan.json').read_text());summary=json.loads((card/'summary.json').read_text())
for relative,digest in extra['native_manifest_hashes'].items():assert hashlib.sha256((root/relative/'qlib_store_manifest.json').read_bytes()).hexdigest()==digest
qlib.init(provider_uri=str(root/plan['provider']),region='us')
with R.uri_context(summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=summary['recorder_id'],experiment_name=card.name);weights=rec.load_object('inputs/weights');stocks=rec.load_object('analysis/native')
core=load_qlib_feature_frame(QlibProviderConfig(root/extra['core_provider'],freq='1min'),instruments=['SOXX'],fields=['$open','$close','$volume','$price_valid'],start=plan['start'],end=plan['as_of']+' 16:00:00')
def groups(native):
 assert native.attrs.get('qlib_data_api')==['qlib.data.D.features'];x=native.reset_index().rename(columns={c:c.removeprefix('$') for c in native.columns});x['date']=x.datetime.dt.normalize();return dict(tuple(x.groupby(['instrument','date'])))
stock_groups,core_groups=groups(stocks),groups(core);rows=[];member_rows=[];empty=pd.DataFrame(columns=['datetime','open','close','volume','price_valid'])
for day,basket in weights.groupby('datetime'):
 w=basket.set_index('instrument').weight_pct;w=w/w.sum()
 for cutoff in extra['cutoffs']:
  returns=pd.Series({symbol:closed_window_return(stock_groups.get((symbol,day),empty),day=day,cutoff=cutoff,frequency_minutes=5) for symbol in w.index});benchmark=closed_window_return(core_groups.get(('SOXX',day),empty),day=day,cutoff=cutoff,frequency_minutes=1);known=bool(returns.notna().all() and np.isfinite(benchmark))
  close_at=day+pd.Timedelta(hours=int(cutoff[:2]),minutes=int(cutoff[3:]));row={'date':day,'year':day.year,'cutoff':cutoff,'window_closed_at':close_at,'assumed_available_at':close_at+pd.Timedelta(minutes=1),'known':known,'members':len(w),'unknown_member_count':int(returns.isna().sum()),'soxx_return':benchmark,'positive_weight':np.nan,'weighted_stock_return':np.nan,'index_up_basket_not_majority':False,'index_down_basket_majority':False}
  if known:
   breadth=float(w.loc[returns.gt(0)].sum());row.update(positive_weight=breadth,weighted_stock_return=float((w*returns).sum()),index_up_basket_not_majority=bool(benchmark>0 and breadth<.5),index_down_basket_majority=bool(benchmark<0 and breadth>.5))
  rows.append(row)
  member_rows.extend({'date':day,'instrument':symbol,'cutoff':cutoff,'weight':float(w.loc[symbol]),'return':float(returns.loc[symbol])} for symbol in w.index)
observations=pd.DataFrame(rows);assert len(observations)==1004
members=pd.DataFrame(member_rows);member_returns=members.pivot(index=['date','instrument'],columns='cutoff',values='return');member_weights=members.loc[members.cutoff.eq('09:45')].set_index(['date','instrument']).weight
transition_rows=[]
for day,part in member_returns.groupby(level='date'):
 w=member_weights.reindex(part.index);known=bool(observations.loc[observations.date.eq(day),'known'].all());up=down=rotation=zero_weight=np.nan
 if known:
  up=float(w.loc[part['09:45'].lt(0)&part['10:30'].gt(0)].sum());down=float(w.loc[part['09:45'].gt(0)&part['10:30'].lt(0)].sum());rotation=2*min(up,down);zero_weight=float(w.loc[part['09:45'].eq(0)|part['10:30'].eq(0)].sum());assert 0<=rotation<=1+1e-12;assert np.isclose(rotation,up+down-abs(up-down))
 transition_rows.append({'date':day,'year':day.year,'window_closed_at':day+pd.Timedelta(hours=10,minutes=30),'assumed_available_at':day+pd.Timedelta(hours=10,minutes=31),'known':known,'strict_up_cross_weight':up,'strict_down_cross_weight':down,'two_way_sign_rotation':rotation,'zero_endpoint_weight':zero_weight})
transitions=pd.DataFrame(transition_rows);assert len(transitions)==502
# Fixed semantic checks: pure one-way repair is zero; opposite member replacement can be nonzero despite zero net crossing.
assert 2*min(.4,0)==0 and np.isclose(2*min(.2,.2),.4)
transition_stats=transitions.groupby('year').agg(calendar_sessions=('known','size'),known_sessions=('known','sum'),nonzero_rotation_sessions=('two_way_sign_rotation',lambda x:x.gt(0).sum()),median_rotation=('two_way_sign_rotation','median'),mean_rotation=('two_way_sign_rotation','mean'),mean_zero_endpoint_weight=('zero_endpoint_weight','mean'))
stats=[]
for (year,cutoff),part in observations.groupby(['year','cutoff']):
 known=part.loc[part.known];disagree=known.index_up_basket_not_majority|known.index_down_basket_majority
 stats.append({'year':year,'cutoff':cutoff,'calendar_sessions':len(part),'known_sessions':len(known),'index_up_basket_not_majority':int(known.index_up_basket_not_majority.sum()),'index_down_basket_majority':int(known.index_down_basket_majority.sum()),'simultaneous_breadth_soxx_rank_correlation':float(known.positive_weight.corr(known.soxx_return,method='spearman')),'simultaneous_weighted_return_soxx_correlation':float(known.weighted_stock_return.corr(known.soxx_return)),'disagreement_abs_soxx_return_bp_median':float(known.loc[disagree,'soxx_return'].abs().median()*1e4)})
daily=load_qlib_feature_frame(QlibProviderConfig(root/extra['daily_provider'],freq='day'),instruments=plan['symbols'],fields=['$open','$close'],start=plan['start'],end=plan['as_of']);daily['daily_intraday_return']=daily['$close']/daily['$open']-1
marks=stocks.reset_index();marks['date']=marks.datetime.dt.normalize();marks=marks.sort_values('datetime').groupby(['instrument','date']).agg(intraday_open=('$open','first'),intraday_close=('$close','last'));marks.index=marks.index.set_names(['instrument','datetime']);marks['minute_intraday_return']=marks.intraday_close/marks.intraday_open-1
comparison=weights.set_index(['instrument','datetime'])[['weight_pct']].join(marks).join(daily[['daily_intraday_return']]);comparison['difference_bp']=(comparison.minute_intraday_return-comparison.daily_intraday_return)*1e4
wp=root/extra['wolf_daily_manifest'];assert hashlib.sha256(wp.read_bytes()).hexdigest()==extra['wolf_daily_manifest_sha256'];wm=json.loads(wp.read_text());wf=wp.parent/'futu_wolf_none_raw.csv';assert hashlib.sha256(wf.read_bytes()).hexdigest()==wm['hashes'][wf.name];wd=pd.read_csv(wf);wd['datetime']=pd.to_datetime(wd.time_key).dt.normalize();wd=wd.set_index('datetime');wmk=marks.xs('WOLF').reindex(weights.loc[weights.instrument.eq('WOLF'),'datetime']);wolf_check=wmk.join(wd[['open','close']]);wolf_check['close_difference']=wolf_check.intraday_close-wolf_check.close;assert wolf_check.close.notna().all();assert wolf_check.close_difference.abs().max()<1e-4
# WOLF was absent from main daily provider. Source identity is independently anchored by old SEC shares/value; this same-vendor daily comparison is repeatability, not independent price discovery.
price_stats=comparison.groupby(level='instrument').agg(active_sessions=('weight_pct','size'),known_daily_comparisons=('difference_bp','count'),max_abs_intraday_return_difference_bp=('difference_bp',lambda x:x.abs().max()),differences_above_1bp=('difference_bp',lambda x:x.abs().gt(1).sum()))
summary['artifact_policy']['recorder_only_artifacts']=['inputs/*','analysis/*']
summary.update(candidate_descriptors=1,candidate_information_qualified=False,wolf_daily_compared_sessions=len(wolf_check),wolf_daily_max_close_difference=float(wolf_check.close_difference.abs().max()),known_snapshots=int(observations.known.sum()),snapshots=len(observations),daily_known_comparisons=int(comparison.difference_bp.notna().sum()),daily_missing_comparisons=int(comparison.difference_bp.isna().sum()),daily_differences_above_1bp=int(comparison.difference_bp.abs().gt(1).sum()))
with R.uri_context(summary['recorder_uri']):
 with R.start(experiment_name=card.name,recorder_id=summary['recorder_id'],resume=True):
  rec=R.get_recorder();rec.save_objects(artifact_path='inputs',observation_plan=extra,quota_after=json.loads((card/'quota_after.json').read_text()),intake_resume=json.loads((card/'intake_resume.json').read_text()),wolf_daily_manifest=wm,wolf_daily_source=wd);rec.save_objects(artifact_path='analysis',snapshots=observations,member_window_returns=members,transitions=transitions,transition_statistics=transition_stats,window_statistics=pd.DataFrame(stats),daily_return_comparison=comparison,wolf_daily_comparison=wolf_check,summary=summary)
  artifacts=Path(summary['recorder_uri'])/summary['experiment_id']/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),known_snapshots=int(observations.known.sum()))
atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=summary['experiment_id']);atomic_json(summary,card/'summary.json');pd.DataFrame(stats).to_csv(card/'window_observability.csv',index=False);price_stats.to_csv(card/'daily_return_comparison.csv');print(pd.DataFrame(stats).to_string(index=False));print({k:summary[k] for k in ['daily_known_comparisons','daily_missing_comparisons','daily_differences_above_1bp']})

import yaml
workflow=json.loads(json.dumps({'experiment':card.name,'provider_uri':summary['provider_uri'],'recorder_uri':summary['recorder_uri'],'artifact_policy':summary['artifact_policy']}))
(card/'workflow.yaml').write_text(yaml.safe_dump(workflow,sort_keys=False))

transition_stats.to_csv(card/'transition_observability.csv');print(transition_stats.to_string())


# 跨期成分分钟来源与固定窗口核查

本轮0训练、0回测、0未来收益筛选。固定2024—2025全部502个交易日，37个历史成员并集，每天使用当日前已披露的30成员。新采35只，AMD/NVDA复用原有完整NONE/RTH来源；共1187592条5分钟观察。采集min/max只是请求边界，RMBS退出期间仍不进入每日篮子。

全天完整有效篮子502/502日，活跃成员缺失bar 0条、无效价量行0条；没有插值或部分篮子归一化。固定09:45/10:30共1004个快照，符合完整前缀、有效价格、首尾有成交条件的1004个。

|年份|切点|可用/全部|加权收益与SOXX同期相关|上涨广度与SOXX同期排序相关|方向分歧日|
|---|---|---|---|---|---|
|2024|09:45|252/252|0.9857|0.9307|29|
|2024|10:30|252/252|0.9938|0.9502|22|
|2025|09:45|250/250|0.9874|0.9177|30|
|2025|10:30|250/250|0.9912|0.9368|28|

跨源校验使用同日开收盘收益比率，避免把不同拆股价格水平直接拼接。主日线可比较14831个成员日，缺失229个，超过1bp差异112个；差异是审查标记，不据此过滤日期或认定某源一定正确。旧WOLF另与已保存、经证券身份锚点核验的旧股日线比较229日，最大收盘差0.00000183美元。该项是同供应商重复性核查，不是独立供应商验证。

采集曾在MCHP发生请求超时，进程已明确退出后降为单连接续采；完成源复用，未完成分页按整日边界续接，原页哈希及替换行数均保留。研究范围和两个切点未变。

决定：historical_basket_source_only_no_alpha_claim。这里的相关性是同时段描述关系，不是未来收益IC。高同步提醒简单汇总重复市场走势的风险，但不证明全部内部信息都无预测价值；分歧也可能来自陈旧披露权重、ETF权重差异或成交时点，不能自动生成交易规则。

防过拟合边界：百万条bar并不等于百万独立样本；本期只有502个交易日，同一天的股票和两个切点都有关联。两个年份已被项目其他研究查看，不能称为全新样本外。未根据收益增删成员、挑季度、换切点或搜索阈值。日频广度、新低收敛和融资压力的已关闭路线不因本次扩样重新获得资格。历史bar闭合后一分钟只是研究可用时点假设，不是实盘首次接收认证；披露篮子不是每日真实指数成分权重。

下一步需事先冻结一个能区别于ETF同期走势的具体机制及唯一信息检验，连同方向、标签时点、比较基线和失败停止条件；信息通过后才考虑完整Tiered、10万美元、独立成本路径与回撤约束。当前没有新增策略收益，未证明稳定净超额，现役模型和生产release未改。

[采集范围](collection_ranges.csv) · [窗口统计](window_observability.csv) · [价格口径比较](daily_return_comparison.csv) · [防过拟合SOP](../../docs/research_failure_to_feature_sop.md)

Recorder `5ba89655e5b14f5e857a906d9108e57f`，原始数据及大证据存NAS，Git保留瘦卡。


跨源开收盘收益差绝对值中位为0.0000bp，99%分位0.8184bp，最大7.5102bp；大于5bp共8个成员日。最大案例UMC在2024-11-18的分钟开盘6.795、主日线开盘6.800美元，收盘均6.94，半美分量级开盘差足以影响低价股的近零符号。尚未认定哪一源错误，不替换开盘价或过滤这些日期。

## 唯一待验证描述：成分双向轮动

同一已披露篮子、固定09:45和10:30，以开盘以来收益严格跨零定义转强U与转弱D的权重。two_way_sign_rotation = 2×min(U,D) = U+D−|U−D|。单向普遍修复为0；同时有等量股票转强与转弱时，即使净广度变化为0，也会显示正轮动。任一切点恰好零收益不算严格跨越，另记零端点权重，不能通过加epsilon改变符号。全篮子或SOXX前缀未知时该描述也未知。

这与简单上涨占比或五日新低收敛的对象不同：它记录具体成员在两个已闭合盘中切点之间的双向替换。它仍可能受零点附近噪声和篮子权重陈旧影响；低轮动也可能只是持续下跌，不能直接解释为应加仓。公式在跨期描述运行前冻结，未查看未来收益、没有训练或参数搜索。

|年份|可用日|轮动大于零日|平均双向轮动权重|平均零端点权重|
|---|---|---|---|---|
|2024|252|124|5.40%|0.57%|
|2025|250|144|6.91%|0.42%|

该字段只取得描述层资格，candidate_information_qualified=false。下一步先做开盘价精度敏感性核查；零点附近的双向替换可能来自计价差，不应立即进入收益筛选。不是已选中有效特征，不替代失败路线，也没有主张未来方向；下一项固定信息检验必须明确条件、主指标和标签时点，不能直接套回测挑窗口。
